# Module 5B: Selection Signatures - Detecting Adaptive Evolution

**An Interactive Journey from Neutral Expectation to Selection Detection**

---

## The Pattern Hunters Approach

**Traditional teaching:** "Tajima's D tests for selection. Here's the formula. Calculate it."

**Our approach:** "What does genetic variation look like under pure drift? Let's observe. Then we'll see how selection creates different patterns!"

---

### Learning Objectives

By the end of this module, you will be able to:

- **Understand** Kimura's neutral theory as the null hypothesis
- **Observe** what site frequency spectrum (SFS) looks like under neutrality
- **Recognize** how different types of selection distort the SFS
- **Calculate** Tajima's D and interpret its values
- **Detect** selection signatures using multiple methods
- **Apply** these concepts to real populations and conservation

---

### Module Structure (120-150 minutes)

**Part 1:** The Neutral Baseline - Kimura's Revolution (30 min)  
**Part 2:** Site Frequency Spectrum - The Shape of Variation (30 min)  
**Part 3:** Tajima's D - Detecting Deviations (30 min)  
**Part 4:** Selection Signatures - Multiple Methods (30 min)  
**Part 5:** Real Examples & Applications (30 min)

---

**Authors:** Dr. Alok Patel & Ms. Susama Kar  
**Institution:** Department of Zoology, Kuchinda College  
**Series:** The Pattern Hunters - Genetics Education  
**DOI:** 10.5281/zenodo.17887470

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from ipywidgets import interact, interactive, FloatSlider, IntSlider, Dropdown, Checkbox
import ipywidgets as widgets
from IPython.display import display, HTML, Markdown

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("✓ All libraries imported successfully!")
print("✓ Ready to explore selection signatures!")

---

# Part 1: The Neutral Baseline - Kimura's Revolution

## The Question That Changed Population Genetics

**1968:** Motoo Kimura asks a radical question:

> "What if MOST genetic changes at the molecular level are neutral - neither helpful nor harmful?"

This was controversial! Everyone assumed genetic changes must be either beneficial (selected for) or harmful (selected against).

---

## 🎓 Three Levels of Understanding

### 9th Grade: The Spelling Mistake Analogy

Imagine copying a long essay. You make some spelling mistakes:

- **"colour" → "color"** - Different spelling, same meaning → **NEUTRAL**
- **"the" → "teh"** - Wrong, confusing → **HARMFUL** (selected against)
- **"good" → "great"** - Better word → **BENEFICIAL** (selected for)

Kimura said: Most DNA "spelling mistakes" are like colour→color. They don't change the "meaning" (protein function), so natural selection doesn't care about them. They just drift randomly!

### BSc Level: The Molecular Reality

**DNA codes for proteins using:**
- Synonymous mutations (silent) - Same amino acid → Usually NEUTRAL
- Nonsynonymous mutations (replacement) - Different amino acid → Often SELECTED

**Example:** 
- Codon UUU → UUC (both code for Phe) = Synonymous
- Codon UUU → UUA (Phe → Leu) = Nonsynonymous

**Kimura's prediction:** If most mutations are neutral, then:
- Synonymous sites should show MORE variation
- Nonsynonymous sites should show LESS variation
- Molecular clock should be approximately constant

### MSc/Research: The Mathematical Framework

**Neutral theory predicts:**

$$\theta = 4N_e\mu$$

Where:
- θ = Expected nucleotide diversity
- Ne = Effective population size
- μ = Mutation rate per generation

**Under neutrality:**
- Rate of substitution = mutation rate (independent of Ne!)
- Site frequency spectrum follows specific shape
- Ratio of polymorphism to divergence is constant

**References:**
- Kimura, M. (1968). Evolutionary rate at molecular level. *Nature* 217:624-626.
- Kimura, M. (1983). *The Neutral Theory of Molecular Evolution*. Cambridge Univ Press.

---

## Interactive 1: Observe Synonymous vs Nonsynonymous Variation

**EDUCATIONAL SIMULATION - Based on real genomic patterns**

Let's verify Kimura's prediction: Are synonymous sites more variable?

We'll simulate a gene and track mutations over time.

In [ ]:
def simulate_neutral_vs_selection(generations=1000, pop_size=100, 
                                  syn_mutation_rate=0.001, nonsyn_mutation_rate=0.001,
                                  selection_coefficient=-0.01):
    """
    Simulate accumulation of synonymous vs nonsynonymous mutations
    
    SIMULATED DATA - Educational demonstration
    Parameters based on realistic evolutionary scenarios
    """
    np.random.seed(42)
    
    # Track diversity over time
    syn_diversity = [0]
    nonsyn_diversity = [0]
    
    # Simulate evolution
    for gen in range(1, generations + 1):
        # Synonymous mutations (neutral)
        syn_mutations = np.random.poisson(pop_size * syn_mutation_rate)
        syn_fixed = syn_mutations  # All eventually drift to fixation or loss
        
        # Nonsynonymous mutations (mostly deleterious)
        nonsyn_mutations = np.random.poisson(pop_size * nonsyn_mutation_rate)
        # Selection removes most deleterious mutations
        # Probability of fixation for deleterious allele ≈ 0 for Nes << -1
        if selection_coefficient < 0:
            removal_prob = 1 - np.exp(2 * selection_coefficient)
            nonsyn_fixed = np.random.binomial(nonsyn_mutations, 1 - removal_prob)
        else:
            nonsyn_fixed = nonsyn_mutations
        
        # Accumulate diversity
        syn_diversity.append(syn_diversity[-1] + syn_fixed * 0.01)  # Scaled for visualization
        nonsyn_diversity.append(nonsyn_diversity[-1] + nonsyn_fixed * 0.01)
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Diversity accumulation over time
    gens = np.arange(0, generations + 1)
    axes[0].plot(gens, syn_diversity, 'b-', linewidth=2.5, label='Synonymous (neutral)', alpha=0.8)
    axes[0].plot(gens, nonsyn_diversity, 'r-', linewidth=2.5, label='Nonsynonymous (selected)', alpha=0.8)
    axes[0].fill_between(gens, 0, syn_diversity, alpha=0.2, color='blue')
    axes[0].fill_between(gens, 0, nonsyn_diversity, alpha=0.2, color='red')
    
    axes[0].set_xlabel('Generations', fontsize=12)
    axes[0].set_ylabel('Nucleotide Diversity (π)', fontsize=12)
    axes[0].set_title('Kimura\'s Prediction: Neutral Sites Vary More', fontsize=14, weight='bold')
    axes[0].legend(fontsize=11, loc='upper left')
    axes[0].grid(True, alpha=0.3)
    
    # Add annotation
    final_ratio = syn_diversity[-1] / nonsyn_diversity[-1] if nonsyn_diversity[-1] > 0 else 999
    axes[0].text(0.98, 0.5, f'Ratio πS/πN = {final_ratio:.1f}',
                transform=axes[0].transAxes, fontsize=11,
                ha='right', va='center',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    # Add SIMULATED label
    axes[0].text(0.02, 0.98, 'SIMULATED DATA', transform=axes[0].transAxes,
                fontsize=10, color='orange', weight='bold', va='top',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
    
    # Plot 2: Final comparison
    categories = ['Synonymous\n(Neutral)', 'Nonsynonymous\n(Selected)']
    final_values = [syn_diversity[-1], nonsyn_diversity[-1]]
    colors = ['blue', 'red']
    
    bars = axes[1].bar(categories, final_values, color=colors, alpha=0.6, 
                      edgecolor='black', linewidth=2)
    axes[1].set_ylabel('Final Nucleotide Diversity (π)', fontsize=12)
    axes[1].set_title('Neutral vs Selected Sites', fontsize=14, weight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, val in zip(bars, final_values):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{val:.1f}', ha='center', va='bottom', 
                    fontsize=12, weight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # Interpretation
    print("\n" + "="*70)
    print("KIMURA'S PREDICTION VERIFIED")
    print("="*70)
    
    print(f"\nAfter {generations} generations:")
    print(f"  • Synonymous diversity (πS): {syn_diversity[-1]:.2f}")
    print(f"  • Nonsynonymous diversity (πN): {nonsyn_diversity[-1]:.2f}")
    print(f"  • Ratio πS/πN: {final_ratio:.2f}")
    
    print("\nWhat does this mean?")
    if final_ratio > 2:
        print("  ✓ Synonymous sites ARE more variable!")
        print("  ✓ This matches Kimura's neutral theory prediction")
        print("  ✓ Selection is removing variation at nonsynonymous sites")
    else:
        print("  ! Ratio is low - increase selection coefficient or generations")
    
    print("\nBiological interpretation:")
    print("  • Synonymous changes don't affect protein → Neutral")
    print("  • They accumulate freely by genetic drift")
    print("  • Nonsynonymous changes affect protein → Often deleterious")
    print("  • Selection removes them before they spread")
    print("  • Result: Synonymous sites show more variation")
    
    print("\n" + "="*70)
    print("KEY INSIGHT: The neutral baseline")
    print("="*70)
    print("\nThis difference between πS and πN is our evidence that:")
    print("  1. Most mutations at molecular level are neutral (Kimura was right!)")
    print("  2. Selection acts on mutations that change protein function")
    print("  3. We can DETECT selection by comparing to neutral expectation")
    print("\n" + "="*70)

# Create interactive widget
interact(simulate_neutral_vs_selection,
         generations=IntSlider(min=100, max=2000, step=100, value=1000,
                              description='Generations:', continuous_update=False),
         pop_size=IntSlider(min=50, max=500, step=50, value=100,
                           description='Pop size:', continuous_update=False),
         syn_mutation_rate=FloatSlider(min=0.0001, max=0.01, step=0.0001, value=0.001,
                                      description='μ (syn):', continuous_update=False, readout_format='.4f'),
         nonsyn_mutation_rate=FloatSlider(min=0.0001, max=0.01, step=0.0001, value=0.001,
                                         description='μ (nonsyn):', continuous_update=False, readout_format='.4f'),
         selection_coefficient=FloatSlider(min=-0.05, max=0.0, step=0.001, value=-0.01,
                                          description='s (nonsyn):', continuous_update=False, readout_format='.3f'));

---

## 💡 Why This Matters: The Neutral Theory as Null Hypothesis

**Kimura's genius:** He gave us a **null hypothesis** for evolution!

### Without Neutral Theory:
❌ See variation → "Maybe selection? Maybe drift? Who knows?"

### With Neutral Theory:
✅ See variation → "What does NEUTRAL look like?"  
✅ Compare observation to neutral expectation  
✅ Deviation = Evidence for selection!  

---

### The Scientific Method Applied:

1. **Null Hypothesis (H₀):** Variation is due to neutral drift
2. **Prediction:** Specific patterns (SFS shape, πS/πN ratio, etc.)
3. **Observation:** Measure actual patterns
4. **Test:** Does observation match neutral prediction?
5. **Conclusion:** 
   - Match → Consistent with neutrality
   - Mismatch → Evidence for selection!

**This is how we detect selection!**

---

# Part 2: Site Frequency Spectrum - The Shape of Variation

## What Is the Site Frequency Spectrum (SFS)?

**Simple definition:** A histogram showing how many mutations exist at different frequencies in a population.

**Example:**
- You sequence 10 individuals (20 chromosomes)
- Find a mutation present in 2 copies → Frequency = 2/20 = 0.10
- Count how many mutations at each frequency
- Plot it!

---

## 🎓 Three Levels of Understanding

### 9th Grade: The Rare Variants Story

Imagine surveying family names in a village:
- **Many rare names** (1-2 families) → New arrivals
- **Few common names** (many families) → Old established families

Under neutral drift:
- Most mutations are RECENT (rare)
- Few mutations are OLD (common)
- This creates a specific shape!

### BSc Level: The Mathematical Shape

**Under neutrality (Watterson 1975):**

Expected number of variants at frequency i:

$$E[\xi_i] = \frac{\theta}{i}$$

Where:
- θ = 4Neμ (population mutation rate)
- i = number of copies (1, 2, 3, ...)

**This predicts:** Many singletons (i=1), fewer doubletons (i=2), even fewer at higher frequencies

### MSc/Research: How Selection Changes the Shape

**Positive selection (selective sweep):**
- Beneficial mutation rises rapidly
- Removes variation nearby
- SFS: Excess of HIGH frequency variants
- Tajima's D > 0

**Purifying selection (background selection):**
- Deleterious mutations removed
- Reduces Ne locally
- SFS: Excess of LOW frequency variants
- Tajima's D < 0

**Population expansion:**
- Many new mutations
- SFS: Excess of rare variants
- Tajima's D < 0

**References:**
- Watterson, G.A. (1975). Number of segregating sites. *Theoretical Population Biology* 7:256-276.
- Tajima, F. (1989). Statistical method for testing neutral mutation hypothesis. *Genetics* 123:585-595.

---

## Interactive 2: Explore the Site Frequency Spectrum

**EDUCATIONAL SIMULATION**

Let's see how different evolutionary scenarios create different SFS shapes!

In [ ]:
def explore_sfs(scenario='Neutral', sample_size=20, theta=10):
    """
    Explore Site Frequency Spectrum under different scenarios
    
    SIMULATED DATA - Educational demonstration
    """
    np.random.seed(42)
    
    # Generate SFS based on scenario
    frequencies = np.arange(1, sample_size)
    
    if scenario == 'Neutral':
        # Neutral expectation: E[ξi] = θ/i
        expected = theta / frequencies
        observed = np.random.poisson(expected)
        color = 'green'
        description = "Under neutrality: Many rare variants, few common variants"
        
    elif scenario == 'Positive Selection (Sweep)':
        # Excess of high-frequency variants
        expected = theta / frequencies
        # Boost high frequencies
        boost = 1 + 2 * (frequencies / sample_size)
        expected = expected * boost
        observed = np.random.poisson(expected)
        color = 'orange'
        description = "Selective sweep: Excess of HIGH frequency variants"
        
    elif scenario == 'Purifying Selection':
        # Excess of low-frequency variants
        expected = theta / frequencies
        # Boost low frequencies
        boost = 1 + 2 * (1 - frequencies / sample_size)
        expected = expected * boost
        observed = np.random.poisson(expected)
        color = 'red'
        description = "Purifying selection: Excess of LOW frequency variants"
        
    elif scenario == 'Population Expansion':
        # Huge excess of singletons
        expected = theta / frequencies
        expected[0] = expected[0] * 3  # Triple singletons
        observed = np.random.poisson(expected)
        color = 'purple'
        description = "Pop expansion: Extreme excess of rare variants"
        
    elif scenario == 'Bottleneck':
        # Loss of rare variants
        expected = theta / frequencies
        expected[0] = expected[0] * 0.3  # Reduce singletons
        observed = np.random.poisson(expected)
        color = 'brown'
        description = "Bottleneck: Fewer rare variants, more intermediate"
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Site Frequency Spectrum
    axes[0].bar(frequencies, observed, color=color, alpha=0.7, edgecolor='black', linewidth=1.5)
    
    # Add neutral expectation line
    neutral_expected = theta / frequencies
    axes[0].plot(frequencies, neutral_expected, 'k--', linewidth=2, 
                label='Neutral expectation', alpha=0.7)
    
    axes[0].set_xlabel('Allele Count (number of copies)', fontsize=12)
    axes[0].set_ylabel('Number of SNPs', fontsize=12)
    axes[0].set_title(f'Site Frequency Spectrum: {scenario}', fontsize=14, weight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Add description
    axes[0].text(0.98, 0.95, description,
                transform=axes[0].transAxes, fontsize=10,
                ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    # Add SIMULATED label
    axes[0].text(0.02, 0.95, 'SIMULATED DATA', transform=axes[0].transAxes,
                fontsize=9, color='orange', weight='bold', va='top',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.3))
    
    # Plot 2: Folded SFS (more common in practice)
    # Fold the spectrum (combine i and n-i)
    folded_freqs = []
    folded_counts = []
    mid = sample_size // 2
    
    for i in range(1, mid + 1):
        if i < sample_size - i:
            count = observed[i-1] + observed[sample_size - i - 1]
        else:
            count = observed[i-1]
        folded_freqs.append(i)
        folded_counts.append(count)
    
    axes[1].bar(folded_freqs, folded_counts, color=color, alpha=0.7, 
               edgecolor='black', linewidth=1.5)
    axes[1].set_xlabel('Minor Allele Count', fontsize=12)
    axes[1].set_ylabel('Number of SNPs', fontsize=12)
    axes[1].set_title('Folded SFS (Minor Allele Frequency)', fontsize=14, weight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Add explanation
    axes[1].text(0.98, 0.95, 'Folded: Can\'t distinguish derived/ancestral',
                transform=axes[1].transAxes, fontsize=10,
                ha='right', va='top',
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    # Calculate Tajima's D (simplified)
    # π (pairwise differences)
    pi = np.sum(observed * frequencies * (sample_size - frequencies)) / (sample_size * (sample_size - 1) / 2)
    
    # θW (Watterson's estimator)
    S = np.sum(observed)  # Number of segregating sites
    a1 = np.sum(1.0 / np.arange(1, sample_size))
    theta_W = S / a1
    
    # Tajima's D (simplified)
    D = (pi - theta_W) / (theta_W * 0.5)  # Simplified variance
    
    print("\n" + "="*70)
    print(f"SCENARIO: {scenario.upper()}")
    print("="*70)
    
    print(f"\nSFS Statistics:")
    print(f"  • Total segregating sites (S): {S}")
    print(f"  • Singletons: {observed[0]} ({observed[0]/S*100:.1f}% of variants)")
    print(f"  • Nucleotide diversity (π): {pi:.2f}")
    print(f"  • Watterson's θ: {theta_W:.2f}")
    print(f"  • Tajima's D (approximate): {D:.2f}")
    
    print(f"\nInterpretation:")
    if D < -1:
        print("  • D << 0: Excess of rare variants")
        print("  • Possible causes: Purifying selection, population expansion, selective sweep")
    elif D > 1:
        print("  • D >> 0: Excess of intermediate-frequency variants")
        print("  • Possible causes: Balancing selection, population bottleneck, structure")
    else:
        print("  • D ≈ 0: Consistent with neutral evolution")
    
    print("\n" + "="*70)

# Create interactive widget
interact(explore_sfs,
         scenario=Dropdown(options=['Neutral', 'Positive Selection (Sweep)', 
                                   'Purifying Selection', 'Population Expansion', 'Bottleneck'],
                          value='Neutral', description='Scenario:'),
         sample_size=IntSlider(min=10, max=50, step=2, value=20,
                              description='Sample (2n):', continuous_update=False),
         theta=FloatSlider(min=5, max=50, step=5, value=10,
                          description='θ (variation):', continuous_update=False));

---

## 🔍 What You Discovered

**Try these experiments:**

1. **Neutral scenario:**
   - See the characteristic 1/i shape
   - Many singletons, few high-frequency variants
   - D ≈ 0

2. **Positive selection:**
   - Flatter spectrum
   - More intermediate/high frequency variants
   - D > 0

3. **Purifying selection:**
   - Even MORE singletons than neutral
   - Steep decline
   - D < 0

4. **Population expansion:**
   - Extreme singleton excess
   - D << 0

**Key insight:** Different evolutionary forces create different SFS shapes!

---

# Part 3: Tajima's D - Detecting Deviations

## What Is Tajima's D?

**Simple explanation:** A test statistic that compares two different ways of measuring genetic diversity.

**The two estimators:**

1. **π (pi)** = Average pairwise differences
   - Sensitive to allele frequencies
   - Weights all variants by frequency

2. **θW (Watterson's theta)** = Based on number of segregating sites
   - Each variant counts equally
   - Independent of frequency

**Under neutrality:** π ≈ θW → Tajima's D ≈ 0

**Under selection:** π ≠ θW → Tajima's D ≠ 0

---

## The Formula

$$D = \frac{\pi - \theta_W}{\sqrt{Var(\pi - \theta_W)}}$$

Where variance depends on sample size and number of segregating sites.

**Interpretation:**
- **D = 0:** Consistent with neutral evolution
- **D < 0:** Excess of rare variants (purifying selection, expansion, sweep)
- **D > 0:** Excess of intermediate variants (balancing selection, bottleneck)

**Significance:**
- |D| > 2: Significant deviation from neutrality (p < 0.05)

---

## Interactive 3: Calculate Tajima's D

**EDUCATIONAL TOOL**

Input sequence data and calculate Tajima's D step-by-step!

In [ ]:
def calculate_tajima_d(num_sequences=10, num_segregating_sites=20, 
                      avg_pairwise_diffs=15, show_steps=True):
    """
    Calculate Tajima's D with detailed explanation
    
    EDUCATIONAL CALCULATOR
    """
    
    n = num_sequences
    S = num_segregating_sites
    
    # Calculate a1 and a2 (Tajima 1989)
    a1 = np.sum(1.0 / np.arange(1, n))
    a2 = np.sum(1.0 / np.arange(1, n)**2)
    
    # b1 and b2
    b1 = (n + 1) / (3 * (n - 1))
    b2 = 2 * (n**2 + n + 3) / (9 * n * (n - 1))
    
    # c1 and c2
    c1 = b1 - 1/a1
    c2 = b2 - (n + 2)/(a1 * n) + a2/(a1**2)
    
    # e1 and e2
    e1 = c1 / a1
    e2 = c2 / (a1**2 + a2)
    
    # Watterson's estimator
    theta_W = S / a1
    
    # Pi (provided)
    pi = avg_pairwise_diffs
    
    # Variance of (pi - theta_W)
    var_D = e1 * S + e2 * S * (S - 1)
    
    # Tajima's D
    if var_D > 0:
        D = (pi - theta_W) / np.sqrt(var_D)
    else:
        D = 0
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: π vs θW comparison
    estimators = ['π\n(Pairwise)', 'θW\n(Segregating)']
    values = [pi, theta_W]
    colors = ['blue' if pi < theta_W else 'red', 'green']
    
    bars = axes[0].bar(estimators, values, color=colors, alpha=0.6, 
                      edgecolor='black', linewidth=2)
    axes[0].set_ylabel('Diversity Estimate', fontsize=12)
    axes[0].set_title('Two Diversity Estimators', fontsize=14, weight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Add values
    for bar, val in zip(bars, values):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{val:.2f}', ha='center', va='bottom', 
                    fontsize=12, weight='bold')
    
    # Add difference arrow
    mid_x = 0.5
    if pi > theta_W:
        axes[0].annotate('', xy=(mid_x, theta_W), xytext=(mid_x, pi),
                        arrowprops=dict(arrowstyle='<->', lw=2, color='red'))
        axes[0].text(mid_x + 0.15, (pi + theta_W)/2, f'Δ = {pi - theta_W:.2f}',
                    fontsize=11, color='red', weight='bold')
    else:
        axes[0].annotate('', xy=(mid_x, pi), xytext=(mid_x, theta_W),
                        arrowprops=dict(arrowstyle='<->', lw=2, color='blue'))
        axes[0].text(mid_x + 0.15, (pi + theta_W)/2, f'Δ = {theta_W - pi:.2f}',
                    fontsize=11, color='blue', weight='bold')
    
    # Plot 2: Tajima's D interpretation
    # Draw a number line
    axes[1].axhline(y=0.5, color='black', linewidth=2)
    
    # Mark zones
    axes[1].axvspan(-3, -2, alpha=0.2, color='blue', label='Purifying/Expansion')
    axes[1].axvspan(-2, 2, alpha=0.2, color='green', label='Neutral')
    axes[1].axvspan(2, 3, alpha=0.2, color='red', label='Balancing/Bottleneck')
    
    # Mark D value
    axes[1].scatter([D], [0.5], s=300, c='gold', edgecolor='black', 
                   linewidth=3, zorder=5, marker='v')
    axes[1].text(D, 0.7, f'D = {D:.2f}', ha='center', fontsize=14, 
                weight='bold', bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))
    
    axes[1].set_xlabel("Tajima's D", fontsize=12)
    axes[1].set_title("Tajima's D Interpretation", fontsize=14, weight='bold')
    axes[1].set_xlim(-3, 3)
    axes[1].set_ylim(0, 1)
    axes[1].set_yticks([])
    axes[1].legend(loc='upper right', fontsize=10)
    axes[1].grid(True, alpha=0.3, axis='x')
    
    # Mark significance thresholds
    axes[1].axvline(x=-2, color='blue', linestyle='--', alpha=0.5)
    axes[1].axvline(x=2, color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    
    if show_steps:
        print("\n" + "="*70)
        print("TAJIMA'S D CALCULATION (STEP-BY-STEP)")
        print("="*70)
        
        print("\nInput data:")
        print(f"  • Number of sequences (n): {n}")
        print(f"  • Segregating sites (S): {S}")
        print(f"  • Average pairwise differences (π): {pi:.2f}")
        
        print("\nStep 1: Calculate Watterson's estimator (θW)")
        print(f"  a₁ = Σ(1/i) for i=1 to n-1 = {a1:.4f}")
        print(f"  θW = S/a₁ = {S}/{a1:.4f} = {theta_W:.2f}")
        
        print("\nStep 2: Compare π and θW")
        print(f"  π = {pi:.2f}")
        print(f"  θW = {theta_W:.2f}")
        print(f"  Difference: π - θW = {pi - theta_W:.2f}")
        
        print("\nStep 3: Calculate variance")
        print(f"  a₂ = {a2:.4f}")
        print(f"  e₁ = {e1:.4f}")
        print(f"  e₂ = {e2:.4f}")
        print(f"  Var(π - θW) = {var_D:.4f}")
        
        print("\nStep 4: Calculate Tajima's D")
        print(f"  D = (π - θW) / √Var")
        print(f"  D = {pi - theta_W:.2f} / {np.sqrt(var_D):.2f}")
        print(f"  D = {D:.3f}")
        
        print("\n" + "="*70)
        print("INTERPRETATION")
        print("="*70)
        
        print(f"\nTajima's D = {D:.3f}")
        
        if D < -2:
            print("\n  ⚠️  D << 0: STRONG negative deviation")
            print("  • Significant excess of rare variants")
            print("  • Possible causes:")
            print("    - Purifying selection removing deleterious mutations")
            print("    - Population expansion (many new mutations)")
            print("    - Recent selective sweep")
        elif D < -1:
            print("\n  ℹ️  D < 0: Negative deviation")
            print("  • Excess of rare variants")
            print("  • Suggests recent population growth or weak selection")
        elif D > 2:
            print("\n  ⚠️  D >> 0: STRONG positive deviation")
            print("  • Significant excess of intermediate-frequency variants")
            print("  • Possible causes:")
            print("    - Balancing selection (heterozygote advantage)")
            print("    - Population bottleneck (rare variants lost)")
            print("    - Population structure (Wahlund effect)")
        elif D > 1:
            print("\n  ℹ️  D > 0: Positive deviation")
            print("  • Excess of intermediate-frequency variants")
            print("  • Might indicate balancing selection or structure")
        else:
            print("\n  ✓ D ≈ 0: Consistent with neutral evolution")
            print("  • No significant deviation from neutral expectation")
            print("  • Pattern matches drift without selection")
        
        print("\n" + "="*70)

# Create interactive widget
interact(calculate_tajima_d,
         num_sequences=IntSlider(min=5, max=50, step=5, value=10,
                                description='n (sequences):', continuous_update=False),
         num_segregating_sites=IntSlider(min=5, max=100, step=5, value=20,
                                        description='S (SNPs):', continuous_update=False),
         avg_pairwise_diffs=FloatSlider(min=5, max=50, step=1, value=15,
                                       description='π (pairwise):', continuous_update=False),
         show_steps=Checkbox(value=True, description='Show detailed steps'));

---

# Part 4: Selection Signatures - Multiple Methods

## Beyond Tajima's D

Tajima's D is just one method! Let's explore others:

### 1. **FST Outliers**
- Calculate FST at many loci across genome
- Most loci: FST reflects neutral drift
- Loci under selection: FST much HIGHER than expected
- **Use:** Detect local adaptation

### 2. **Extended Haplotype Homozygosity (EHH/iHS)**
- Selection sweeps create long haplotypes
- Neutral alleles: short haplotypes (broken by recombination)
- Selected alleles: extended haplotypes
- **Use:** Detect recent positive selection

### 3. **dN/dS Ratio**
- Compare nonsynonymous (dN) to synonymous (dS) substitutions
- dN/dS < 1: Purifying selection
- dN/dS = 1: Neutral
- dN/dS > 1: Positive selection
- **Use:** Molecular evolution studies

### 4. **McDonald-Kreitman Test**
- Compare polymorphism vs divergence
- Tests for adaptive evolution
- **Use:** Detect adaptation in protein-coding genes

---

## Real Examples from Indian Populations

### Example 1: Lactase Persistence (Human)

**Published data:** Basu et al. (2016) *Genome Biology* 17:166

**Finding:**
- European populations: Strong selection signature at LCT gene
- Indian populations: WEAK or absent signature
- Why? Different dietary history (less dairy dependence)

**Methods showing signal:**
- iHS scores: High in Europeans, low in Indians
- FST: High between populations at LCT
- Interpretation: Recent selection in Europe, not India

### Example 2: Malaria Resistance (Human)

**Published data:** Various studies

**Finding:**
- G6PD deficiency alleles common in malaria-endemic regions
- Balancing selection maintains variation
- Tajima's D > 0 at G6PD locus

**Tribal populations in Odisha:**
- Higher frequency of protective alleles
- FST outlier between tribal and non-tribal
- Evidence of local adaptation

### Example 3: Thermal Adaptation (Labeo rohita)

**Hypothetical application to your research:**

**Scenario:**
- Compare populations from cool (Mahanadi upper) vs warm (lower) regions
- Scan genome for FST outliers
- Find high FST at heat-shock protein genes

**Interpretation:**
- Local adaptation to temperature
- Selection maintains different alleles in different environments
- Important for aquaculture (don't mix populations!)

---

# Summary & Key Takeaways

## What We Learned

### 1. The Neutral Foundation
- Kimura's neutral theory provides null hypothesis
- Most molecular changes are neutral
- Synonymous sites vary more than nonsynonymous
- This is our baseline for comparison

### 2. The Site Frequency Spectrum
- Shows distribution of allele frequencies
- Under neutrality: Many rare, few common variants
- Selection changes this pattern
- Different forces create different shapes

### 3. Tajima's D
- Compares π and θW
- D = 0: Neutral
- D < 0: Excess rare variants (purifying/expansion/sweep)
- D > 0: Excess intermediate variants (balancing/bottleneck)

### 4. Multiple Methods
- FST outliers detect local adaptation
- EHH/iHS detect recent sweeps
- dN/dS for molecular evolution
- Use multiple methods for robust inference

### 5. Real Applications
- Human adaptation (lactase, malaria resistance)
- Conservation (local adaptation matters)
- Aquaculture (thermal adaptation)
- Medicine (drug resistance evolution)

---

## The Big Picture

**Selection detection workflow:**

1. **Establish neutral expectation** (Kimura's theory)
2. **Calculate test statistics** (Tajima's D, FST, etc.)
3. **Compare to neutral** (deviation = selection?)
4. **Consider alternatives** (demography can mimic selection)
5. **Use multiple methods** (convergent evidence is strong)
6. **Functional validation** (does it affect phenotype?)

---

## Practice Problems

**Problem 1:** A gene shows πS = 0.015 and πN = 0.003. What does this suggest?

**Problem 2:** Tajima's D = -2.5 genome-wide. What could explain this?

**Problem 3:** One locus has FST = 0.45 while genome average is FST = 0.08. Interpretation?

**Problem 4:** You find dN/dS = 1.8 for a gene. What type of selection?

---

## Further Reading

**Classic Papers:**
- Kimura, M. (1968). Evolutionary rate at molecular level. *Nature* 217:624-626.
- Tajima, F. (1989). Statistical method for testing. *Genetics* 123:585-595.
- McDonald, J.H. & Kreitman, M. (1991). Adaptive evolution. *Nature* 351:652-654.
- Sabeti, P.C. et al. (2007). Genome-wide detection. *Nature* 449:913-918.

**Reviews:**
- Nielsen, R. (2005). Molecular signatures of selection. *Annu Rev Genet* 39:197-218.
- Vitti, J.J. et al. (2013). Detecting selection. *Mol Biol Evol* 30:1703-1720.

**Indian Examples:**
- Basu, A. et al. (2016). Indian population genomics. *Genome Biology* 17:166.
- Reich, D. et al. (2009). Reconstructing Indian history. *Nature* 461:489-494.

---

## Next Steps

**Continue learning:**
- **Module 5C:** Effective population size estimation
- Apply to your own data!
- Explore genome-wide scans

---

**Congratulations!** 🎉

You've completed Module 5B and can now:
- Understand neutral theory as foundation
- Calculate and interpret Tajima's D
- Recognize selection signatures
- Apply multiple detection methods
- Interpret real genomic data

**Keep exploring the patterns!** 🔬🧬